In [36]:
import os
# Load .env file
USER_AGENT = os.getenv("USER_AGENT")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [65]:
#build trivia now
from pydantic import BaseModel
class State(BaseModel):
    question:list[str]
    correct_answers: str
    wrong_answer: int

def question_node(state:State) :
    for q in state.question:
        print(f"ques: {q}")
all_questions = "what is the capital of nigeria","who is the president of nigeria","what is the fathers role"
State=(all_questions)


#this project is a trivia question and expect and answer 

In [ ]:
from langchain_openai import ChatOpenAI
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# Define the state structure
class State(TypedDict):
    query: str
    answer: str
    
#node 1
def llm_node(state:State):
    llm_reponse = llm.invoke( f"provide response to the query{state["query"]}")
    state["answer"]= llm_reponse.content
    print("provide the first response")
    return state

def check_response(state:State):
    llm_correct_response = llm.invoke(f"check if the answer provided is correct {state['answer']}")
    state["answer"] = llm_correct_response.content
    print("checked and it is correct")
    return state

graph = StateGraph(State)
graph.add_node("llm_node", llm_node)  # Node to generate LLM response
graph.add_node("check_response", check_response)  # Node to check the response

# Add edges
graph.add_edge(START, "llm_node")  # Start with the LLM node
graph.add_edge("llm_node", "check_response")  # Check the response after generating it
graph.add_edge("check_response", END)  # End after checking the response

builder = graph.compile()
user_query= input("ask anything")
ini_state = State(query =user_query,answer="")
builder.invoke(ini_state)

 

provide the first response
checked and it is correct


{'query': 'what is the capital of nigeria',
 'answer': 'Yes, that answer is correct. The capital of Nigeria is Abuja.'}

In [42]:
from langchain_openai import ChatOpenAI
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# Define the state structure
class State(TypedDict):
    query: str
    sentiment: str
 
# Define nodes
def negative_node(state: State) -> State:
    print("Negative feedback detected.")
    return state

def positive_node(state: State) -> State:
    print("Positive feedback detected.")
    return state
#create a function like a sentiment function
def sentiment_node(state:State):
    response =llm.invoke(  f"Analyze the sentiment of the following review and respond with only 'positive' or 'negative': {state['query']}"
    )

    state["sentiment"] = response.content
    return state 

# Define conditional edge logic
# def feedback(state: State) -> str:
#     if state["sentiment"] == 'positive':
#         print("This is a positive feedback.")
#         return "p"  # Go to positive_node
#     else:
#         print("This is a negative feedback.")
#         return "n"  # Go to negative_node

# Build the graph
graph = StateGraph(State)

# Add nodes
graph.add_node("sentiment_node", sentiment_node)
graph.add_node("p", positive_node)  # "p" is the name of the positive node
graph.add_node("n", negative_node)  # "n" is the name of the negative node

# Add conditional edges
graph.add_edge(START, "sentiment_node")
graph.add_conditional_edges(START,
    sentiment_node,  # Start from the beginning
    # feedback,  # Use the feedback function to decide the next node
)

# Add edges
graph.add_edge("p", END)  # After positive_node, end the graph
graph.add_edge("n", END)  # After negative_node, end the graph

# Compile the graph
app = graph.compile()

# Run the graph
initial_state = State(query=input("Enter your feedback: "))
app.invoke(initial_state)


{'query': "This is the worst product I've ever used. Terrible experience.",
 'sentiment': 'negative'}

In [37]:
from langchain_openai import ChatOpenAI
llm_model = ChatOpenAI(model="gpt-4o-mini")
llm = llm_model.invoke("my name is doris")
print(llm)

content='Nice to meet you, Doris! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 12, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_06737a9306', 'finish_reason': 'stop', 'logprobs': None} id='run-61c0f3d3-728c-4f9d-8ddc-6ee358e685a3-0' usage_metadata={'input_tokens': 12, 'output_tokens': 15, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [38]:

from typing import Literal
from typing_extensions import TypedDict

class State(TypedDict):
    query: str 

In [39]:
#node 1
def question(state):
    state["query"] = "markurdi"
    

In [40]:
#node2
def ans(state):
    state["query"] = "otukpo"

In [41]:
#conditional edge
faq = "what is the capital of benue state" 
def check(state):
   state["query"] == faq
   if state["query"] == question:
      return question
   else: 
       return ans

In [42]:
from langgraph.graph import StateGraph, START, END

graph =StateGraph(State)
graph.add_node("question", question),
graph.add_node("ans" , ans)
graph.add_node("check", check)
graph.add_conditional_edges("check",check)
graph.add_edge(START,"question")
graph.add_edge(START,"ans")
graph.add_edge("question", END)
graph.add_edge("ans",END)
builder = graph.compile()
print(builder)

In [ ]:
# Define the state structure
class State(TypedDict):
    query: str  # Node 1

# Define the functions that modify the state
def question(state: State) -> State:
    state["query"] = "markurdi"
    return state

def ans(state: State) -> State:
    state["query"] = "otukpo"
    return state

# Define the conditional function
def check(state: State) -> Literal["question", "ans"]:
    faq = "what is the capital of benue state"
    if state["query"] == faq:
        return "question"
    else:
        return "ans"

# Create the graph
graph = StateGraph(State)

# Add nodes to the graph
graph.add_node("question", question)
graph.add_node("ans", ans)
graph.add_node("check", check)

# Add edges to the graph
graph.add_edge(START, "check")  # Start at the "check" node
graph.add_conditional_edges("check", check)  # Conditional edge based on the "check" function
graph.add_edge("question", END)  # End after "question"
graph.add_edge("ans", END)  # End after "ans"

# Compile the graph
builder = graph.compile()
# Print the result


ValueError: Invalid input type <class 'langgraph.graph.state.CompiledStateGraph'>. Must be a PromptValue, str, or list of BaseMessages.